# 🚀 Qwen3.8-27B Abliterated (Heretic) Vision GGUF - Colab Server & Endpoint

This ready-to-run notebook sets up **Qwen3.8-27B-Heretic-Abliterated-Uncensored-GGUF** (`Q4_K_M` + Vision `mmproj`) on Google Colab with:
- **High-Speed Model Download** via `aria2c` (16 connections, downloads in ~2 mins)
- **llama-server with CUDA acceleration** & automatic layer-offload detection (T4 / L4 / A100)
- **Multimodal Vision Projector** (`mmproj-Qwen3.8-27B-Q8_0.gguf`) enabled for image/video reasoning
- **Active Keep-Alive** (dual-layer JavaScript + Web Audio heartbeat to prevent disconnects)
- **Instant Public HTTPS Endpoint** via Cloudflare Tunnel (`trycloudflare.com`)
- **Permanent Endpoint Integration** with Cloudflare Zero Trust Named Tunnel or Cloudflare Worker Proxy

### ⚡ Cell 1: Colab Keep-Alive (Prevents Idle Disconnections)
Run this cell first. It injects an automated JavaScript watcher and a background audio heartbeat so Colab does not disconnect your session.

In [ ]:
# @title 🛡️ Enable Colab Keep-Alive
from IPython.display import display, HTML

keep_alive_html = '''
<div style="padding:12px; border-radius:8px; background:#1e1e2f; color:#a6e3a1; font-family:monospace;">
  <b>✅ Keep-Alive Active:</b> Sending periodic UI interaction pings & silent audio heartbeat every 60s.
</div>
<script>
function keepAlive() {
    console.log("[Keep-Alive] Pinging Colab runtime at " + new Date().toLocaleTimeString());
    // 1. Simulate button interaction
    const btn = document.querySelector("colab-connect-button") || 
                document.querySelector("colab-toolbar-button#connect") ||
                document.querySelector("#top-toolbar > colab-connect-button");
    if (btn) btn.click();
    
    // 2. Audio context heartbeat to keep browser tab active
    try {
        const ctx = new (window.AudioContext || window.webkitAudioContext)();
        const osc = ctx.createOscillator();
        const gain = ctx.createGain();
        gain.gain.value = 0.0001; // Silent
        osc.connect(gain);
        gain.connect(ctx.destination);
        osc.start();
        osc.stop(ctx.currentTime + 0.1);
    } catch(e) {}
}
setInterval(keepAlive, 60000);
</script>
'''
display(HTML(keep_alive_html))


### ⚙️ Cell 2: System Setup & GPU Detection
Detects available VRAM, installs `aria2` for ultra-fast multi-threaded downloading, and configures `llama.cpp` with CUDA.

In [ ]:
# @title 🔧 Install Dependencies & Build/Download llama.cpp
import os
import subprocess
import sys

# 1. Verify GPU
gpu_info = subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
print(f"Detected GPU: {gpu_info}")
vram_mb = int(subprocess.getoutput("nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits").split()[0])

# 2. Install aria2 and cloudflared
!apt-get update -qq && apt-get install -y -qq aria2
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 3. Set up llama.cpp with CUDA
if not os.path.exists("llama.cpp"):
    print("Cloning and building llama.cpp with CUDA support (takes ~90 seconds)... ")
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
    !cd llama.cpp && cmake -B build -DGGML_CUDA=ON && cmake --build build --config Release -j$(nproc) --target llama-server
else:
    print("llama.cpp already built!")

LLAMA_SERVER_BIN = "/content/llama.cpp/build/bin/llama-server"
if not os.path.exists(LLAMA_SERVER_BIN):
    # Fallback path if directory structure varies
    LLAMA_SERVER_BIN = "/content/llama.cpp/llama-server"

print(f"llama-server binary located at: {LLAMA_SERVER_BIN}")


### 📥 Cell 3: Fast Model & Vision Projector Download
Downloads `Qwen3.8-27B-Heretic-Q4_K_M.gguf` (~16.8 GB) and `mmproj-Qwen3.8-27B-Q8_0.gguf` (~800 MB) from `0bserverx/Qwen3.8-27B-Heretic-Abliterated-Uncensored-GGUF`.

In [ ]:
# @title 📦 Download Model Weights (Q4_K_M + Vision mmproj)
import os

MODEL_REPO = "0bserverx/Qwen3.8-27B-Heretic-Abliterated-Uncensored-GGUF"
MODEL_FILE = "Qwen3.8-27B-Heretic-Q4_K_M.gguf"
MMPROJ_FILE = "mmproj-Qwen3.8-27B-Q8_0.gguf"

MODEL_URL = f"https://huggingface.co/{MODEL_REPO}/resolve/main/{MODEL_FILE}"
MMPROJ_URL = f"https://huggingface.co/{MODEL_REPO}/resolve/main/{MMPROJ_FILE}"

print("Starting ultra-fast download via aria2 (16 connections)... ")
if not os.path.exists(MODEL_FILE):
    !aria2c -x 16 -s 16 -k 1M "{MODEL_URL}" -o "{MODEL_FILE}"
else:
    print(f"{MODEL_FILE} already downloaded!")

if not os.path.exists(MMPROJ_FILE):
    !aria2c -x 8 -s 8 -k 1M "{MMPROJ_URL}" -o "{MMPROJ_FILE}"
else:
    print(f"{MMPROJ_FILE} already downloaded!")

print("✅ Download complete! Files verified:")
!ls -lh *.gguf


### 🚀 Cell 4: Launch `llama-server` (OpenAI-compatible Vision & Text API)
Launches `llama-server` on port `8080`.
- On **L4 / A100 (>=22GB VRAM)**: Offloads **100% of layers** (`-ngl 99`) to GPU.
- On **T4 (15.3GB VRAM)**: Offloads **32 layers** to GPU and keeps the rest in system RAM (`-ngl 32`) to prevent Out-Of-Memory (OOM) errors.

In [ ]:
# @title 🤖 Start llama-server (Background Daemon)
import subprocess
import time
import os

# Determine GPU layers based on VRAM
vram_mb = int(subprocess.getoutput("nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits").split()[0])
if vram_mb >= 22000:
    gpu_layers = 99  # Full GPU offload for L4/A100
    ctx_size = 8192
else:
    gpu_layers = 30  # Optimized split for T4 (15GB VRAM + 12GB CPU RAM)
    ctx_size = 4096

print(f"Configuring llama-server with -ngl {gpu_layers} (VRAM: {vram_mb} MB, Context: {ctx_size} tokens)...")

# Kill any existing server process
!pkill -f llama-server || true

API_KEY = 'sk-qwen-colab-2026'

cmd = [
    LLAMA_SERVER_BIN,
    "-m", MODEL_FILE,
    "--mmproj", MMPROJ_FILE,
    "-ngl", str(gpu_layers),
    "-c", str(ctx_size),
    "--host", "0.0.0.0",
    "--port", "8080",
    "--alias", "qwen3.8-27b",
    "--api-key", API_KEY,
    "--flash-attn",
    "--chat-template", "chatml"
]

log_file = open("server.log", "w")
server_proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)
print(f"llama-server spawned with PID {server_proc.pid}. Waiting for weights to load into VRAM...")

# Poll until ready
for i in range(60):
    time.sleep(2)
    res = subprocess.getoutput("curl -s http://127.0.0.1:8080/health || true")
    if '"status":"ok"' in res or '"status": "ok"' in res or '"status":"loading model"' in res:
        if '"status":"ok"' in res or '"status": "ok"' in res:
            print("\n🎉 llama-server is healthy and ready for inference!")
            break
        else:
            print(".", end="", flush=True)
    else:
        print(".", end="", flush=True)
else:
    print("\nCheck server.log for details:")
    !tail -n 25 server.log


### 🌐 Cell 5: Create Public HTTPS Endpoint (Cloudflare Tunnel)
Exposes your local `llama-server` to the internet.

- **Mode 1 (Quick Free Tunnel - No signup required):** Leave `TUNNEL_TOKEN` blank. Generates an instant `https://*.trycloudflare.com` URL.
- **Mode 2 (Permanent Domain via Cloudflare Zero Trust):** Paste your free Cloudflare Tunnel token below. Your endpoint will be permanently bound to your custom domain (e.g., `https://qwen.yourdomain.com`).
- **Mode 3 (Auto-Register with Cloudflare Worker):** If you use the Cloudflare Worker proxy provided in your workspace, supply the `WORKER_URL` and `WORKER_SECRET` below so Colab auto-updates the Worker on every restart!

In [ ]:
# @title 🔗 Launch Cloudflare Tunnel & Expose API
import subprocess
import time
import re
import requests

# Optional: Set your Cloudflare Zero Trust Tunnel Token for a fixed permanent domain
TUNNEL_TOKEN = ""  # @param {type:"string"}

# Optional: Set your Cloudflare Worker URL to auto-update its upstream target
WORKER_URL = ""     # @param {type:"string"}
WORKER_SECRET = ""  # @param {type:"string"}

!pkill -f cloudflared || true

if TUNNEL_TOKEN.strip():
    print("Starting Named Cloudflare Tunnel with your permanent token...")
    !cloudflared tunnel run --token "{TUNNEL_TOKEN.strip()}" > tunnel.log 2>&1 &
    print("✅ Permanent Tunnel running! Connect using your configured custom domain.")
else:
    print("Starting Quick Free Cloudflare Tunnel (trycloudflare.com)...")
    !cloudflared tunnel --url http://127.0.0.1:8080 > tunnel.log 2>&1 &
    
    public_url = None
    for _ in range(30):
        time.sleep(1)
        log_content = subprocess.getoutput("cat tunnel.log")
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)
        if match:
            public_url = match.group(0)
            break
            
    if public_url:
        print("\n=======================================================")
        print(f"🎯 Public HTTPS Endpoint: {public_url}")
        print(f"📡 OpenAI Base URL:      {public_url}/v1")
        print(f"🔑 API Key:              sk-qwen-colab-2026")
        print(f"🧠 Model Name:           qwen3.8-27b")
        print(f"💬 Web UI Interface:     {public_url}")
        print("=======================================================\n")
        
        # Auto-register with Cloudflare Worker if configured
        if WORKER_URL.strip():
            try:
                reg_res = requests.post(
                    f"{WORKER_URL.strip().rstrip('/')}/register",
                    json={"backend_url": public_url, "secret": WORKER_SECRET.strip()},
                    timeout=5
                )
                print(f"⚡ Auto-registered with Cloudflare Worker: {reg_res.text}")
            except Exception as e:
                print(f"Failed to register with Worker: {e}")
    else:
        print("Failed to extract tunnel URL. Log output:")
        !cat tunnel.log


### 🧪 Cell 6: Test Vision & Text Inference (OpenAI Compatible)
Verifies that both text generation and multimodal vision reasoning are functioning properly.

In [ ]:
# @title 🖼️ Test Chat Completion with Vision
import requests
import json

# Sample public image of a cat
test_image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg"

payload = {
    "model": "qwen3.8-27b",
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe what you see in this image in 2 sentences, then state your uncensored opinion of cats."},
                {"type": "image_url", "image_url": {"url": test_image_url}}
            ]
        }
    ],
    "max_tokens": 300,
    "temperature": 0.7
}

response = requests.post("http://127.0.0.1:8080/v1/chat/completions", json=payload)
print("Response Status:", response.status_code)
try:
    result = response.json()
    print("\n--- Model Output ---")
    print(result['choices'][0]['message']['content'])
except Exception as e:
    print("Raw Response:", response.text)
